# Ornith Heretic Kaggle API

Minimal Kaggle launcher for the 9B Ornith-1.5 Heretic GGUF using `llama.cpp`, with an authenticated OpenAI-compatible API and optional Cloudflare Quick Tunnel.

Enable Kaggle Internet and a GPU accelerator before running. The notebook downloads the selected public GGUF automatically; no model Dataset attachment is required.

In [ ]:
# ===== CONFIGURATION =====
MODEL_REPO = "Thunder13240/Ornith-1.5-9B-heretic-GGUF"
MODEL_FILENAME = "Ornith-1.5-9B-heretic-IQ4_XS.gguf"
MODEL_DIR = "/kaggle/working/models"
LLAMA_CPP_DIR = "/kaggle/working/llama.cpp"
LLAMA_CPP_REF = "master"
CONTEXT_SIZE = 2048
GPU_LAYERS = 999
SERVER_HOST = "127.0.0.1"
SERVER_PORT = 8080
START_CLOUDFLARE_TUNNEL = True

import sys, time, subprocess, secrets, re
from pathlib import Path


## 1. Download the selected model
Checks the live Hugging Face repository file list first, then downloads the best available quant from the configured preference order. No manually attached Kaggle Dataset is required.

In [ ]:
try:
    from huggingface_hub import HfApi, hf_hub_download
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)
    from huggingface_hub import HfApi, hf_hub_download

# Resolve against the live repo listing instead of assuming a filename that may have changed.
repo_files = HfApi().list_repo_files(repo_id=MODEL_REPO, repo_type="model")
gguf_names = [name for name in repo_files if name.lower().endswith(".gguf") and not Path(name).name.lower().startswith("mmproj-")]
if not gguf_names:
    raise FileNotFoundError(f"No text-model GGUF files found in Hugging Face repo: {MODEL_REPO}")

available_by_lower = {name.lower(): name for name in gguf_names}
requested = MODEL_FILENAME.lower()
if requested in available_by_lower:
    selected_filename = available_by_lower[requested]
else:
    # Prefer a strong quality/size balance; fall back only to files that actually exist.
    quant_preferences = [
        "ornith-1.5-9b-heretic-iq4_xs.gguf",
        "ornith-1.5-9b-heretic-q4_k_m.gguf",
        "ornith-1.5-9b-heretic-iq4_nl.gguf",
        "ornith-1.5-9b-heretic-q4_k_s.gguf",
        "ornith-1.5-9b-heretic-iq3_m.gguf",
        "ornith-1.5-9b-heretic-iq3_xs.gguf",
        "ornith-1.5-9b-heretic-iq2_m.gguf",
    ]
    selected_filename = next((available_by_lower[name] for name in quant_preferences if name in available_by_lower), None)
    if selected_filename is None:
        raise FileNotFoundError(
            "The configured GGUF is missing and no supported fallback quant was found. Available text GGUFs: "
            + ", ".join(sorted(Path(name).name for name in gguf_names))
        )
    print(f"Requested file {MODEL_FILENAME!r} is not present; using available file {selected_filename!r} instead.")

print("Downloading:", selected_filename)
model_path = Path(hf_hub_download(
    repo_id=MODEL_REPO,
    filename=selected_filename,
    local_dir=MODEL_DIR,
))
if not model_path.is_file():
    raise FileNotFoundError(f"Model download did not produce {model_path}")
print("Model ready:", model_path)
print("Model size GiB:", round(model_path.stat().st_size / (2 ** 30), 2))


## 2. Build upstream llama.cpp with CUDA
Builds the server for the Kaggle GPU. The cell explicitly points CMake at CUDA's driver stub when Kaggle's runtime does not expose `CUDA::cuda_driver` automatically.

In [ ]:
llama_dir = Path(LLAMA_CPP_DIR)
if not (llama_dir / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", LLAMA_CPP_REF, "https://github.com/ggml-org/llama.cpp.git", str(llama_dir)], check=True)
else:
    subprocess.run(["git", "-C", str(llama_dir), "fetch", "--depth", "1", "origin", LLAMA_CPP_REF], check=True)
    subprocess.run(["git", "-C", str(llama_dir), "checkout", "FETCH_HEAD"], check=True)
LLAMA_CPP_COMMIT = subprocess.run(["git", "-C", str(llama_dir), "rev-parse", "HEAD"], check=True, capture_output=True, text=True).stdout.strip()
print("llama.cpp commit:", LLAMA_CPP_COMMIT)

cuda_roots = [Path("/usr/local/cuda"), Path("/usr/local/cuda-12.8")]
cuda_driver_stub = None
for root in cuda_roots:
    for relative in ("targets/x86_64-linux/lib/stubs/libcuda.so", "lib64/stubs/libcuda.so"):
        candidate = root / relative
        if candidate.is_file():
            cuda_driver_stub = candidate
            break
    if cuda_driver_stub:
        break

build_dir = llama_dir / "build"
if (build_dir / "CMakeCache.txt").exists():
    (build_dir / "CMakeCache.txt").unlink()

cmake_args = [
    "cmake", "-S", str(llama_dir), "-B", str(build_dir),
    "-DGGML_CUDA=ON", "-DLLAMA_BUILD_SERVER=ON",
    "-DCMAKE_BUILD_TYPE=Release", "-DCUDAToolkit_ROOT=/usr/local/cuda",
]
if cuda_driver_stub:
    cmake_args.append("-DCUDA_cuda_driver_LIBRARY=" + str(cuda_driver_stub))
    print("Using CUDA driver stub for linking:", cuda_driver_stub)
else:
    print("CUDA driver stub not found in standard locations; CMake will search the toolkit defaults.")

subprocess.run(cmake_args, check=True)
subprocess.run(["cmake", "--build", str(build_dir), "--config", "Release", "-j", "2", "--target", "llama-server"], check=True)
server_binary = build_dir / "bin" / "llama-server"
if not server_binary.is_file():
    raise FileNotFoundError(f"Build completed but llama-server was not found at {server_binary}")
print("Built:", server_binary)


## 3. Start the local authenticated API
Loads the downloaded IQ4_XS model with a 2K context and full GPU offload. If the model cannot fit or the GPU backend fails, the cell prints the server log and stops before opening a public tunnel.

In [ ]:
try:
    import requests
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "requests"], check=True)
    import requests

API_KEY = secrets.token_urlsafe(32)
server_log_path = Path("/kaggle/working/llama-server.log")
server_log = server_log_path.open("w")
server_cmd = [str(server_binary), "-m", str(model_path), "--host", SERVER_HOST, "--port", str(SERVER_PORT), "-c", str(CONTEXT_SIZE), "-ngl", str(GPU_LAYERS), "--api-key", API_KEY]
print("Starting llama-server; model loading can take several minutes.")
server_process = subprocess.Popen(server_cmd, stdout=server_log, stderr=subprocess.STDOUT, start_new_session=True)
healthy = False
for _ in range(180):
    if server_process.poll() is not None:
        break
    try:
        response = requests.get(f"http://{SERVER_HOST}:{SERVER_PORT}/health", timeout=3)
        if response.ok:
            healthy = True
            break
    except requests.RequestException:
        pass
    time.sleep(5)
if not healthy:
    server_log.flush()
    print(server_log_path.read_text(errors="replace")[-12000:])
    raise RuntimeError("llama-server did not become healthy. Cloudflare tunnel was not started.")
print("Local health check passed.")
print("Server log:", server_log_path)
print("API key created; it is stored in notebook memory and not printed.")


## 4. Test chat completions locally
The API key is sent in the request header and is not printed.

In [ ]:
test_response = requests.post(
    f"http://{SERVER_HOST}:{SERVER_PORT}/v1/chat/completions",
    headers={"Authorization": f"Bearer {API_KEY}"},
    json={"model": "local-model", "messages": [{"role": "user", "content": "Reply with exactly: API test passed"}], "max_tokens": 32},
    timeout=300,
)
test_response.raise_for_status()
test_json = test_response.json()
print(test_json.get("choices", [{}])[0].get("message", {}).get("content", "No message content returned"))

## 5. Optionally expose the API through Cloudflare Quick Tunnel
Only run this after the local health check and chat-completion test pass. Quick Tunnel URLs are temporary and change when the tunnel restarts.

In [ ]:
if START_CLOUDFLARE_TUNNEL:
    import platform, urllib.request
    arch = platform.machine().lower()
    if arch not in ("x86_64", "amd64"):
        raise RuntimeError(f"Unsupported architecture for this cloudflared download cell: {arch}")
    cloudflared = Path("/kaggle/working/cloudflared")
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", cloudflared)
    cloudflared.chmod(0o755)
    tunnel_log_path = Path("/kaggle/working/cloudflared.log")
    tunnel_log = tunnel_log_path.open("w")
    tunnel_process = subprocess.Popen([str(cloudflared), "tunnel", "--no-autoupdate", "--url", f"http://{SERVER_HOST}:{SERVER_PORT}"], stdout=tunnel_log, stderr=subprocess.STDOUT, start_new_session=True)
    public_url = None
    for _ in range(60):
        time.sleep(2)
        tunnel_log.flush()
        tunnel_text = tunnel_log_path.read_text(errors="replace")
        match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", tunnel_text)
        if match:
            public_url = match.group(0)
            break
        if tunnel_process.poll() is not None:
            break
    if not public_url:
        print(tunnel_log_path.read_text(errors="replace")[-8000:])
        raise RuntimeError("Cloudflare Quick Tunnel did not provide a public URL.")
    print("Public API base URL:", public_url)
    print("Use this API key from your client; do not paste it into public logs or commit it to GitHub.")
else:
    print("Tunnel disabled.")

## Client configuration notes
- OpenAI-compatible base URL: append `/v1` to the public tunnel URL.
- API key: the value held in the notebook variable `API_KEY` for this session.
- The notebook must remain running for the endpoint to remain available.
- This is an experimental endpoint. Do not use it for sensitive data. Verify client protocol compatibility and keep the API key private.